# ZimTech Retail Holdings — Sales Intelligence Analysis
**Assessment:** INJIVA-DS-DA-PANDAS-001 · **Data:** 20 fictional transactions, January 2026, USD

This notebook walks through Stages 1–10 of the assessment. All processing logic lives in the reusable modules in `src/`;
the notebook calls those functions and displays/interprets their results, so it does not depend on hidden state.
Run it top-to-bottom with **Kernel → Restart & Run All**.

> No visualisation libraries are used (assessment restriction). Results are shown as DataFrames and written interpretations.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.source_data import sales_data, get_sales_data, get_product_reference, OPERATING_CITIES
from src import validation, cleaning, analytics
from src.pipeline import run_pipeline

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.3


## Stage 1 — DataFrame construction and schema validation

In [2]:
# 11. Verify all dictionary lists have consistent lengths BEFORE building the DataFrame
record_count = validation.check_list_lengths(sales_data)
print("All lists have length:", record_count)

# 3. Build sales_df from a deep copy so the original dictionary is preserved
sales_df = validation.build_dataframe(get_sales_data())
raw_backup = sales_df.copy()          # independent copy kept for auditing/comparison
sales_df                                # 4. display the complete DataFrame

All lists have length: 20


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0
5,TX006,2026-01-10,LAPTOP,Computers,1,700.00,520,Mary,Harare,Bank Transfer,5
6,TX007,2026-01-11,Mouse,Accessories,15,15.00,7,Peter,Gweru,Cash,0
7,TX008,12/01/2026,Smartphone,Mobile,4,320.00,240,Sarah,Mutare,EcoCash,10
8,TX009,2026-01-13,monitor,Computers,-2,200.00,145,John,Bulawayo,Cash,0
9,TX010,2026-01-14,Keyboard,Accessories,8,30.00,14,Mary,Harare,Bank Transfer,0


In [3]:
# 5-7. Shape, column names, dtypes
print("Rows:", sales_df.shape[0], "| Columns:", sales_df.shape[1])
print("Columns:", sales_df.columns.tolist())
sales_df.dtypes

Rows: 20 | Columns: 11
Columns: ['Transaction_ID', 'Date', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Unit_Cost', 'Salesperson', 'City', 'Payment_Method', 'Discount_Percent']


Transaction_ID          str
Date                    str
Product                 str
Category                str
Quantity              int64
Unit_Price          float64
Unit_Cost             int64
Salesperson             str
City                    str
Payment_Method          str
Discount_Percent      int64
dtype: object

In [4]:
# 8. First five and last five records
display(sales_df.head(5))
display(sales_df.tail(5))

,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
15,TX016,2026-01-20,Monitor,Computers,4,190.00,140,Sarah,Mutare,EcoCash,0
16,TX017,2026-01-21,Keyboard,Accessories,9,28.00,13,Peter,Gweru,Cash,0
17,TX018,2026-01-22,Printer,Office Equipment,2,230.00,170,Mary,Harare,Bank Transfer,5
18,TX019,2026-01-23,Smartphone,Mobile,6,330.00,250,Sarah,Mutare,EcoCash,10
19,TX020,2026-01-24,Mouse,Accessories,20,15.00,7,Peter,Gweru,Cash,0


In [5]:
# 9-10. Info summary and memory usage
sales_df.info()
print("\nDeep memory usage (bytes):")
print(sales_df.memory_usage(deep=True))
print("Total:", sales_df.memory_usage(deep=True).sum(), "bytes")

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    20 non-null     str    
 1   Date              20 non-null     str    
 2   Product           20 non-null     str    
 3   Category          20 non-null     str    
 4   Quantity          20 non-null     int64  
 5   Unit_Price        19 non-null     float64
 6   Unit_Cost         20 non-null     int64  
 7   Salesperson       20 non-null     str    
 8   City              20 non-null     str    
 9   Payment_Method    20 non-null     str    
 10  Discount_Percent  20 non-null     int64  
dtypes: float64(1), int64(3), str(7)
memory usage: 1.8 KB

Deep memory usage (bytes):
Index                132
Transaction_ID      1080
Date                1180
Product             1125
Category            1181
Quantity             160
Unit_Price           160
Unit_Cost            160
Salesperson

In [6]:
# 12-13. Mandatory columns and expected logical types
pd.DataFrame({"Column": validation.MANDATORY_COLUMNS,
              "Expected_Type": [validation.EXPECTED_SCHEMA[c] for c in validation.MANDATORY_COLUMNS]})

,Column,Expected_Type
0,Transaction_ID,string
1,Date,datetime
2,Product,string
3,Category,string
4,Quantity,integer
5,Unit_Price,numeric
6,Unit_Cost,numeric
7,Salesperson,string
8,City,string
9,Payment_Method,string


In [7]:
# 14. Reusable schema validation function
schema_report = validation.validate_schema(sales_df)
schema_report

,Column,Expected_Type,Actual_Dtype,Compatible,Missing_Values,Note
0,Transaction_ID,string,str,True,0,
1,Date,datetime,str,True,0,stored as text - conversion to datetime required
2,Product,string,str,True,0,
3,Category,string,str,True,0,
4,Quantity,integer,int64,True,0,
5,Unit_Price,numeric,float64,True,1,numeric but contains missing values
6,Unit_Cost,numeric,int64,True,0,
7,Salesperson,string,str,True,0,
8,City,string,str,True,0,
9,Payment_Method,string,str,True,0,


In [8]:
# 15. Meaningful exception when a required column is missing (a missing COLUMN, not a missing value)
broken = sales_df.drop(columns=["Unit_Cost"])
try:
    validation.validate_schema(broken)
except validation.SchemaValidationError as exc:
    print("SchemaValidationError raised as expected ->", exc)

SchemaValidationError raised as expected -> Missing mandatory column(s): ['Unit_Cost']


**Interpretation — Stage 1.** The dictionary converts cleanly into 20 rows × 11 columns and every list has the same length, so the structure is sound.
But a DataFrame that builds successfully is not necessarily valid: `Date` is stored as text (not datetime), and `Unit_Price` was inferred as `float64` only because one value is missing.
The schema validator separates these *structural* checks from business rules, raises a `SchemaValidationError` naming the missing column, and never relies on column positions or a hardcoded row count.
`raw_backup` and the untouched `sales_data` dictionary preserve the original for auditing.

## Stage 2 — Data quality investigation

In [9]:
# 1-2. Missing values and percentages
missing = pd.DataFrame({"Missing_Count": sales_df.isna().sum(),
                        "Missing_Pct": (sales_df.isna().mean() * 100).round(2)})
missing[missing["Missing_Count"] > 0]

,Missing_Count,Missing_Pct
Unit_Price,1,5.00


In [10]:
# 3-4. Duplicate business identifiers and completely duplicated records
print("Duplicate Transaction_IDs:", sales_df["Transaction_ID"].duplicated().sum())
print("Completely duplicated rows:", sales_df.duplicated().sum())
# Repeated product names are NOT duplicates - many sales of the same product are expected
sales_df["Product"].value_counts()

Duplicate Transaction_IDs: 0
Completely duplicated rows: 0


Product
Smartphone    4
Laptop        3
Printer       3
Mouse         3
Monitor       2
Keyboard      2
keyboard      1
LAPTOP        1
monitor       1
Name: count, dtype: int64

In [11]:
# 5-9. Quantity, price and cost checks (Boolean indexing)
display(sales_df[sales_df["Quantity"] < 0])
display(sales_df[sales_df["Quantity"] == 0])
display(sales_df[sales_df["Unit_Price"].isna()])
print("Non-positive prices:", (sales_df["Unit_Price"] <= 0).sum(),
      "| Price below cost:", (sales_df["Unit_Price"] < sales_df["Unit_Cost"]).sum(),
      "| Invalid costs:", (sales_df["Unit_Cost"].isna() | (sales_df["Unit_Cost"] < 0)).sum())

,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
8,TX009,2026-01-13,monitor,Computers,-2,200.00,145,John,Bulawayo,Cash,0


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
14,TX015,19-01-2026,Laptop,Computers,0,670.00,505,John,Bulawayo,Cash,10


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
11,TX012,16/01/2026,Printer,Office Equipment,2,NaN,165,Sarah,Mutare,EcoCash,0


Non-positive prices: 0 | Price below cost: 0 | Invalid costs: 0


In [12]:
# 10-11, 15. Product names, categories and product-category relationships
print(sorted(sales_df["Product"].unique()))
print(sorted(sales_df["Category"].unique()))
pd.crosstab(sales_df["Product"].str.title(), sales_df["Category"])

['Keyboard', 'LAPTOP', 'Laptop', 'Monitor', 'Mouse', 'Printer', 'Smartphone', 'keyboard', 'monitor']
['Accessories', 'Computers', 'Mobile', 'Office Equipment']


Category,Accessories,Computers,Mobile,Office Equipment
Product,,,,
Keyboard,3,0,0,0
Laptop,0,4,0,0
Monitor,0,3,0,0
Mouse,3,0,0,0
Printer,0,0,0,3
Smartphone,0,0,4,0


In [13]:
# 12-14. Date formats, dtypes and discounts
print(sales_df["Date"].str.extract(r"^(\d{4}-\d{2}-\d{2})$")[0].isna().sum(), "dates are not ISO formatted")
print("Date dtype:", sales_df["Date"].dtype, "| Unit_Price dtype:", sales_df["Unit_Price"].dtype)
sales_df["Discount_Percent"].value_counts().sort_index()

5 dates are not ISO formatted
Date dtype: str | Unit_Price dtype: float64


Discount_Percent
0     11
5      5
10     4
Name: count, dtype: int64

In [14]:
# Structured, rule-based Data Quality Report
quality_report, rule_summary = validation.assess_data_quality(sales_df)
rule_summary

,Rule,Description,Violations,Status
0,missing_values,Missing values in any column other than Unit_P...,0,PASS
1,duplicate_transaction_id,Transaction_ID must be unique,0,PASS
2,duplicate_record,No completely duplicated rows,0,PASS
3,negative_quantity,Quantity must not be negative,1,FAIL
4,zero_quantity,Quantity must not be zero,1,FAIL
5,non_numeric_quantity,Quantity must be numeric,0,PASS
6,missing_unit_price,Unit_Price must be present,1,FAIL
7,invalid_unit_price,Unit_Price must be numeric and > 0,0,PASS
8,price_below_cost,Unit_Price should not be below Unit_Cost,0,PASS
9,invalid_unit_cost,Unit_Cost must be numeric and >= 0,0,PASS


In [15]:
quality_report

,Transaction_ID,Column,Original_Value,Issue,Severity,Recommended_Action,Resolution_Status
0,TX009,Quantity,-2,"Negative quantity - possible return, reversal ...",High,Quarantine for manual review; confirm with sou...,Open
1,TX015,Quantity,0,"Zero quantity - no goods sold, record has no f...",Medium,Quarantine for manual review (possible cancell...,Open
2,TX012,Unit_Price,<missing>,Missing selling price - revenue cannot be dete...,High,Do not impute or zero-fill; quarantine and obt...,Open
3,TX003,Product,keyboard,Inconsistent product naming (standard form: 'K...,Low,Standardise to title case,Open
4,TX006,Product,LAPTOP,Inconsistent product naming (standard form: 'L...,Low,Standardise to title case,Open
5,TX009,Product,monitor,Inconsistent product naming (standard form: 'M...,Low,Standardise to title case,Open
6,TX002,Date,06/01/2026,"Non-standard date format (day-first, not ISO 8...",Low,Parse with explicit day-first format and store...,Open
7,TX005,Date,09-01-2026,"Non-standard date format (day-first, not ISO 8...",Low,Parse with explicit day-first format and store...,Open
8,TX008,Date,12/01/2026,"Non-standard date format (day-first, not ISO 8...",Low,Parse with explicit day-first format and store...,Open
9,TX012,Date,16/01/2026,"Non-standard date format (day-first, not ISO 8...",Low,Parse with explicit day-first format and store...,Open


**Interpretation — Stage 2.** 18 repeatable rules were run; 12 pass with zero violations, which is reported explicitly in the rule summary. The real problems are:
* **TX009** — quantity −2 (Monitor). This may be a return or reversal, but there is no link to an original sale, so it can't be netted safely.
* **TX015** — quantity 0 (Laptop): no goods sold, so the record has no financial effect.
* **TX012** — missing selling price (Printer). Revenue cannot be determined, and the value must **not** be treated as zero.
* 3 inconsistent product names (`keyboard`, `LAPTOP`, `monitor`) and 5 day-first dates (DD/MM/YYYY or DD-MM-YYYY).

There are no duplicate IDs or duplicate rows. Repeated product names are normal repeat sales, not duplicates. All discounts are 0%, 5% or 10%, and every product maps to exactly one expected category.

## Stage 3 — Data cleaning and standardisation
The documented business rules (A1–A7) are in the docstring of `src/cleaning.py`.

In [16]:
print(cleaning.__doc__)


Deterministic, reusable data cleaning and standardisation functions.

Documented business rules / assumptions
---------------------------------------
A1  Product names are standardised by stripping whitespace and converting to
    title case ("LAPTOP", "laptop" -> "Laptop").
A2  Dates are parsed ONLY with these explicit formats, in this order:
        %Y-%m-%d  (ISO, e.g. 2026-01-05)
        %d/%m/%Y  (day-first, e.g. 06/01/2026 -> 6 January 2026)
        %d-%m-%Y  (day-first, e.g. 09-01-2026 -> 9 January 2026)
    Day-first is assumed because (a) the company is Zimbabwean, where DD/MM/YYYY
    is standard, (b) the brief states the data covers January 2026, and (c) the
    resulting dates run in strict sequence with the Transaction_IDs. The
    alternative month-first reading would place these sales in June, September,
    December etc., which contradicts the brief. A value that matches no format
    is rejected, never guessed.
A3  Missing Unit_Price values are NOT imputed (no zero-fi

In [17]:
# Individual reusable functions
print(cleaning.standardise_product_names(pd.Series(["LAPTOP", " keyboard", "monitor"])).tolist())
print(cleaning.parse_dates(["2026-01-05", "06/01/2026", "09-01-2026"]).tolist())

['Laptop', 'Keyboard', 'Monitor']
[Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-06 00:00:00'), Timestamp('2026-01-09 00:00:00')]


In [18]:
result = cleaning.clean_sales_data(sales_df)
quality_report = cleaning.update_resolution_status(quality_report, result)
result.summary

,Metric,Value
0,Raw records,20
1,Accepted records,17
2,Rejected records (manual review),3
3,Product names standardised,3
4,Dates converted from non-ISO formats,5
5,Unparseable dates,0
6,Missing unit prices (not imputed),1
7,Negative quantities,1
8,Zero quantities,1
9,Duplicate Transaction_IDs after cleaning,0


In [19]:
# Rejected records - quarantined for manual review, NOT silently deleted
result.rejected[["Transaction_ID", "Original_Date", "Original_Product", "Quantity",
                 "Unit_Price", "Unit_Cost", "Discount_Percent", "Review_Reason", "Record_Status"]]

,Transaction_ID,Original_Date,Original_Product,Quantity,Unit_Price,Unit_Cost,Discount_Percent,Review_Reason,Record_Status
0,TX009,2026-01-13,monitor,-2,200.00,145.00,0.00,Negative quantity - possible return/reversal; ...,Rejected - manual review required
1,TX012,16/01/2026,Printer,2,NaN,165.00,0.00,Missing unit price - not imputed,Rejected - manual review required
2,TX015,19-01-2026,Laptop,0,670.00,505.00,10.00,Zero quantity - no goods sold,Rejected - manual review required


In [20]:
# Audit trail: original vs standardised values for changed records
accepted = result.accepted
accepted.loc[accepted["Product_Name_Changed"] | accepted["Date_Reformatted"],
             ["Transaction_ID", "Original_Product", "Product", "Original_Date", "Date"]]

,Transaction_ID,Original_Product,Product,Original_Date,Date
1,TX002,Smartphone,Smartphone,06/01/2026,2026-01-06
2,TX003,keyboard,Keyboard,2026-01-07,2026-01-07
4,TX005,Printer,Printer,09-01-2026,2026-01-09
5,TX006,LAPTOP,Laptop,2026-01-10,2026-01-10
7,TX008,Smartphone,Smartphone,12/01/2026,2026-01-12


In [21]:
# Validate the cleaned dataset and confirm no duplicates / original untouched
print("Business rules passed:", validation.validate_cleaned_dataset(accepted))
print("Duplicate IDs after cleaning:", accepted["Transaction_ID"].duplicated().sum())
print("Original DataFrame unchanged:", sales_df.equals(raw_backup))
accepted.dtypes

Business rules passed: True
Duplicate IDs after cleaning: 0
Original DataFrame unchanged: True


Transaction_ID                  string
Date                    datetime64[us]
Product                         string
Category                        string
Quantity                         int64
Unit_Price                     float64
Unit_Cost                      float64
Salesperson                     string
City                            string
Payment_Method                  string
Discount_Percent               float64
Original_Product                string
Original_Date                   string
Product_Name_Changed           boolean
Date_Reformatted                  bool
dtype: object

**Interpretation — Stage 3.** Cleaning is deterministic and reusable. Product names are title-cased. Dates are parsed only with three explicit formats, with day-first chosen because the brief says January 2026 and the dates then run in order with the IDs. Numeric types are corrected.
17 records are accepted and 3 are **quarantined, not deleted** (TX009, TX012, TX015), each with a written reason. Nothing was imputed. `Original_Product` and `Original_Date` keep the audit trail, no duplicates were introduced, the original DataFrame is unchanged, and the accepted data passes every business rule.

## Stage 4 — Financial data transformation

In [22]:
financial_df = analytics.add_financial_columns(accepted)
financial_df[["Transaction_ID", "Product", "Quantity", "Unit_Price", "Discount_Percent", "Gross_Revenue",
              "Discount_Amount", "Net_Revenue", "Total_Cost", "Gross_Profit", "Gross_Profit_Margin"]]

,Transaction_ID,Product,Quantity,Unit_Price,Discount_Percent,Gross_Revenue,Discount_Amount,Net_Revenue,Total_Cost,Gross_Profit,Gross_Profit_Margin
0,TX001,Laptop,2,650.00,0.00,"1,300.00",0.00,"1,300.00","1,000.00",300.00,23.08
1,TX002,Smartphone,5,300.00,5.00,"1,500.00",75.00,"1,425.00","1,100.00",325.00,22.81
2,TX003,Keyboard,10,25.00,0.00,250.00,0.00,250.00,120.00,130.00,52.00
3,TX004,Monitor,3,180.00,10.00,540.00,54.00,486.00,390.00,96.00,19.75
4,TX005,Printer,2,220.00,0.00,440.00,0.00,440.00,320.00,120.00,27.27
5,TX006,Laptop,1,700.00,5.00,700.00,35.00,665.00,520.00,145.00,21.80
6,TX007,Mouse,15,15.00,0.00,225.00,0.00,225.00,105.00,120.00,53.33
7,TX008,Smartphone,4,320.00,10.00,"1,280.00",128.00,"1,152.00",960.00,192.00,16.67
8,TX010,Keyboard,8,30.00,0.00,240.00,0.00,240.00,112.00,128.00,53.33
9,TX011,Laptop,3,680.00,5.00,"2,040.00",102.00,"1,938.00","1,530.00",408.00,21.05


In [23]:
# 1-4. Totals, overall margin (total GP / total NR) and average transaction value
sales_summary = analytics.financial_summary(financial_df)
sales_summary

,Metric,Value
0,Valid_Transactions,17.00
1,Units_Sold,111.00
2,Gross_Revenue,"14,033.00"
3,Discount_Amount,692.50
4,Net_Revenue,"13,340.50"
5,Total_Cost,"10,085.00"
6,Gross_Profit,"3,255.50"
7,Overall_Gross_Profit_Margin_Pct,24.40
8,Average_Transaction_Value,784.74
9,Effective_Discount_Rate_Pct,4.93


In [24]:
# Why the overall margin must NOT be the mean of transaction margins
print("Correct overall margin  :", round(financial_df["Gross_Profit"].sum() / financial_df["Net_Revenue"].sum() * 100, 4))
print("Wrong (mean of margins) :", round(financial_df["Gross_Profit_Margin"].mean(), 4))

Correct overall margin  : 24.4031
Wrong (mean of margins) : 32.8295


In [25]:
# 5. Loss-making transactions
loss_making = financial_df[financial_df["Gross_Profit"] < 0]
print("Loss-making transactions:", len(loss_making))
loss_making

Loss-making transactions: 0


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent,Original_Product,Original_Date,Product_Name_Changed,Date_Reformatted,Gross_Revenue,Discount_Amount,Net_Revenue,Total_Cost,Gross_Profit,Gross_Profit_Margin,Is_Loss_Making


In [26]:
# 6-8. Validate discounts, division-by-zero handling and independent verification
print("Discount amount never exceeds gross revenue:",
      bool((financial_df["Discount_Amount"] <= financial_df["Gross_Revenue"]).all()))
print("safe_margin(50, 0) ->", analytics.safe_margin(50, 0), "(NaN, no ZeroDivisionError)")
print("Independent row-by-row verification passed:", analytics.verify_financial_calculations(financial_df))

Discount amount never exceeds gross revenue: True
safe_margin(50, 0) -> nan (NaN, no ZeroDivisionError)
Independent row-by-row verification passed: True


In [27]:
# Financial effect of rejected records (shown, not hidden)
rej = result.rejected.copy()
rej["Gross_Revenue_If_Booked"] = rej["Quantity"] * rej["Unit_Price"]
rej["Cost_If_Booked"] = rej["Quantity"] * rej["Unit_Cost"]
rej[["Transaction_ID", "Product", "Quantity", "Unit_Price", "Unit_Cost", "Gross_Revenue_If_Booked", "Cost_If_Booked", "Review_Reason"]]

,Transaction_ID,Product,Quantity,Unit_Price,Unit_Cost,Gross_Revenue_If_Booked,Cost_If_Booked,Review_Reason
0,TX009,Monitor,-2,200.00,145.00,-400.00,-290.00,Negative quantity - possible return/reversal; ...
1,TX012,Printer,2,NaN,165.00,<NA>,330.00,Missing unit price - not imputed
2,TX015,Laptop,0,670.00,505.00,0.00,0.00,Zero quantity - no goods sold


**Interpretation — Stage 4.** Across the 17 valid transactions:
* gross revenue $14,033.00, discounts $692.50, so **net revenue is $13,340.50**;
* cost $10,085.00, so **gross profit is $3,255.50**;
* the **overall gross profit margin is 24.40%**, calculated as total profit ÷ total net revenue. The mean of the individual margins would give a misleading 32.83%.

Average transaction value is $784.74. No transaction is loss-making.
Effect of the rejected records: if TX009 were a genuine return it would reduce revenue by $400 and gross profit by $110. TX012's revenue is unknown: at the observed Printer prices of $220–$230 it would be roughly $440–$460 (≈3.3% of net revenue), but this is shown only as an indication and is not booked. TX015 has no financial effect.
This is gross profit, not net profit: operating expenses are not in the data.

## Stage 5 — Advanced sales performance analysis
**Metric definitions:** *Revenue* = Net_Revenue (after discount). *Profit* = Gross_Profit. *Margin* = group Gross_Profit ÷ group Net_Revenue.
*Volume* = units sold. *Transactions* = count of valid transactions. Ties are ranked with `method="min"` (tied items share the better rank).

In [28]:
# 1-4. Company KPIs
kpi = sales_summary.set_index("Metric")["Value"]
kpi[["Net_Revenue", "Valid_Transactions", "Units_Sold", "Average_Transaction_Value"]]

Metric
Net_Revenue                 13,340.50
Valid_Transactions              17.00
Units_Sold                     111.00
Average_Transaction_Value      784.74
Name: Value, dtype: float64

In [29]:
# 5. Five highest-revenue transactions (nlargest)
financial_df.nlargest(5, "Net_Revenue")[["Transaction_ID", "Product", "City", "Salesperson", "Net_Revenue", "Gross_Profit"]]

,Transaction_ID,Product,City,Salesperson,Net_Revenue,Gross_Profit
9,TX011,Laptop,Bulawayo,John,"1,938.00",408.00
15,TX019,Smartphone,Mutare,Sarah,"1,782.00",282.00
11,TX014,Smartphone,Harare,Mary,"1,472.50",297.50
1,TX002,Smartphone,Harare,Mary,"1,425.00",325.00
0,TX001,Laptop,Bulawayo,John,"1,300.00",300.00


In [30]:
# 6-7. Products by revenue and by profit (groupby + agg via group_performance, sort_values, rank)
product_perf = analytics.group_performance(financial_df, "Product")
display(product_perf[["Product", "Units_Sold", "Net_Revenue", "Revenue_Rank"]])
product_perf.sort_values("Gross_Profit", ascending=False)[["Product", "Gross_Profit", "Gross_Profit_Margin_Pct", "Profit_Rank"]]

,Product,Units_Sold,Net_Revenue,Revenue_Rank
0,Smartphone,20,"5,831.50",1
1,Laptop,6,"3,903.00",2
2,Monitor,7,"1,246.00",3
3,Printer,4,877.00,4
4,Keyboard,27,742.00,5
5,Mouse,47,741.00,6


,Product,Gross_Profit,Gross_Profit_Margin_Pct,Profit_Rank
0,Smartphone,"1,096.50",18.80,1
1,Laptop,853.00,21.85,2
5,Mouse,400.00,53.98,3
4,Keyboard,393.00,52.96,4
2,Monitor,296.00,23.76,5
3,Printer,217.00,24.74,6


In [31]:
# 8. Highest-volume categories
category_perf = analytics.group_performance(financial_df, "Category")
category_perf.sort_values("Units_Sold", ascending=False)[["Category", "Units_Sold", "Transactions", "Net_Revenue", "Gross_Profit"]]

,Category,Units_Sold,Transactions,Net_Revenue,Gross_Profit
2,Accessories,74,6,"1,483.00",793.00
0,Mobile,20,4,"5,831.50","1,096.50"
1,Computers,13,5,"5,149.00","1,149.00"
3,Office Equipment,4,2,877.00,217.00


In [32]:
# 9-10, 14. City revenue, profit and contribution (Masvingo shown as unknown, not zero)
city_perf = analytics.city_performance(financial_df, OPERATING_CITIES)
city_perf[["City", "Transactions", "Net_Revenue", "Gross_Profit", "Gross_Profit_Margin_Pct",
           "Revenue_Share_Pct", "Revenue_Rank", "Profit_Rank", "Data_Note"]]

,City,Transactions,Net_Revenue,Gross_Profit,Gross_Profit_Margin_Pct,Revenue_Share_Pct,Revenue_Rank,Profit_Rank,Data_Note
0,Harare,5.00,"4,239.50",992.50,23.41,31.78,1.00,1.00,Records available
1,Mutare,4.00,"4,134.00",794.00,19.21,30.99,2.00,3.00,Records available
2,Bulawayo,3.00,"3,724.00",804.00,21.59,27.91,3.00,2.00,Records available
3,Gweru,5.00,"1,243.00",665.00,53.50,9.32,4.00,4.00,Records available
4,Masvingo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No records in sample - activity unknown


In [33]:
# 11-12. Salespeople
sp_perf = analytics.group_performance(financial_df, "Salesperson")
sp_perf[["Salesperson", "Transactions", "Units_Sold", "Net_Revenue", "Gross_Profit",
         "Gross_Profit_Margin_Pct", "Revenue_Rank", "Profit_Rank"]]

,Salesperson,Transactions,Units_Sold,Net_Revenue,Gross_Profit,Gross_Profit_Margin_Pct,Revenue_Rank,Profit_Rank
0,Mary,5,21,"4,239.50",992.50,23.41,1,1
1,Sarah,4,16,"4,134.00",794.00,19.21,2,3
2,John,3,8,"3,724.00",804.00,21.59,3,2
3,Peter,5,66,"1,243.00",665.00,53.50,4,4


In [34]:
# 13. Most frequently used payment method (transaction count, not revenue)
financial_df["Payment_Method"].value_counts()

Payment_Method
Cash             7
EcoCash          5
Bank Transfer    5
Name: count, dtype: Int64

In [35]:
# 15. Loss-making transactions (query)
financial_df.query("Gross_Profit < 0")

,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent,Original_Product,Original_Date,Product_Name_Changed,Date_Reformatted,Gross_Revenue,Discount_Amount,Net_Revenue,Total_Cost,Gross_Profit,Gross_Profit_Margin,Is_Loss_Making


In [36]:
# 16. Average discount by category (simple mean of Discount_Percent) and effective rate
category_perf.assign(Effective_Discount_Pct=category_perf["Discount_Amount"] / category_perf["Gross_Revenue"] * 100)[
    ["Category", "Avg_Discount_Pct", "Effective_Discount_Pct"]]

,Category,Avg_Discount_Pct,Effective_Discount_Pct
0,Mobile,7.50,7.58
1,Computers,4.00,3.58
2,Accessories,0.00,0.00
3,Office Equipment,2.50,2.56


In [37]:
# 17. Products with below-average gross profit margin (vs overall company margin)
overall_margin = kpi["Overall_Gross_Profit_Margin_Pct"]
print(f"Overall margin: {overall_margin:.2f}%")
analytics.below_average_margin_products(product_perf, overall_margin)[["Product", "Net_Revenue", "Gross_Profit_Margin_Pct"]]

Overall margin: 24.40%


,Product,Net_Revenue,Gross_Profit_Margin_Pct
0,Smartphone,"5,831.50",18.80
1,Laptop,"3,903.00",21.85
2,Monitor,"1,246.00",23.76


In [38]:
# 18. Salespeople below the average salesperson revenue (query with @variable)
analytics.below_average_salespeople(sp_perf)[["Salesperson", "Net_Revenue", "Average"]]

,Salesperson,Net_Revenue,Average
3,Peter,"1,243.00","3,335.12"


In [39]:
# transform(): each transaction's share of its city's revenue, keeping the original index
financial_df["City_Revenue"] = financial_df.groupby("City")["Net_Revenue"].transform("sum")
financial_df["Share_Of_City_Revenue_Pct"] = financial_df["Net_Revenue"] / financial_df["City_Revenue"] * 100
financial_df[["Transaction_ID", "City", "Net_Revenue", "City_Revenue", "Share_Of_City_Revenue_Pct"]].head(8)

,Transaction_ID,City,Net_Revenue,City_Revenue,Share_Of_City_Revenue_Pct
0,TX001,Bulawayo,"1,300.00","3,724.00",34.91
1,TX002,Harare,"1,425.00","4,239.50",33.61
2,TX003,Gweru,250.00,"1,243.00",20.11
3,TX004,Bulawayo,486.00,"3,724.00",13.05
4,TX005,Mutare,440.00,"4,134.00",10.64
5,TX006,Harare,665.00,"4,239.50",15.69
6,TX007,Gweru,225.00,"1,243.00",18.10
7,TX008,Mutare,"1,152.00","4,134.00",27.87


In [40]:
# Reconcile every grouped table to the transaction-level totals
for name, table in {"product": product_perf, "category": category_perf, "city": city_perf.dropna(subset=["Net_Revenue"]),
                    "salesperson": sp_perf}.items():
    print(name, analytics.reconcile_totals(table, financial_df))

product True
category True
city True
salesperson True


In [41]:
# Important observation: each salesperson works in exactly one city
pd.crosstab(financial_df["Salesperson"], financial_df["City"])

City,Bulawayo,Gweru,Harare,Mutare
Salesperson,,,,
John,3,0,0,0
Mary,0,0,5,0
Peter,0,5,0,0
Sarah,0,0,0,4


**Interpretation — Stage 5.**
* **Products:** Smartphone leads on both revenue ($5,831.50, 43.7%) and gross profit ($1,096.50), but its margin (18.8%) is the lowest because it carries 69% of all discounts. Laptop is second on revenue and profit. Mouse and Keyboard sell the most units but make little revenue, at margins above 50%. Ranking by quantity alone would therefore be misleading.
* **Categories:** Mobile has the most revenue, but Computers has the most **gross profit** ($1,149.00, margin 22.3%). Accessories has by far the highest volume (74 units).
* **Cities:** Harare leads on both revenue ($4,239.50) and profit ($992.50). Mutare is second on revenue but third on profit (lowest margin, 19.2%). Gweru has the highest margin (53.5%) but the lowest revenue. Masvingo has **no records**, so its activity is *unknown*, not zero.
* **Salespeople:** Mary leads on revenue and profit. Peter is below average on revenue but has the highest margin.
* **Payment:** Cash is the most-used method (7 of 17 transactions).
* **Critical caveat:** each salesperson sold in exactly one city (Mary–Harare, Sarah–Mutare, John–Bulawayo, Peter–Gweru). City and salesperson effects are therefore **fully confounded** and can't be separated with this data.

All grouped totals reconcile to $13,340.50.

## Stage 6 — Time-series sales analysis

In [42]:
# 1. Day, month, year
ts_df = analytics.add_date_parts(financial_df).sort_values("Date")
ts_df[["Transaction_ID", "Date", "Day", "Month", "Year", "Weekday", "Net_Revenue"]].head()

,Transaction_ID,Date,Day,Month,Year,Weekday,Net_Revenue
0,TX001,2026-01-05,5,1,2026,Monday,"1,300.00"
1,TX002,2026-01-06,6,1,2026,Tuesday,"1,425.00"
2,TX003,2026-01-07,7,1,2026,Wednesday,250.00
3,TX004,2026-01-08,8,1,2026,Thursday,486.00
4,TX005,2026-01-09,9,1,2026,Friday,440.00


In [43]:
# 2, 5, 6, 8. Daily revenue on a full calendar index, cumulative revenue, rolling averages, counts
daily = analytics.daily_revenue(financial_df)
daily

,Date,Net_Revenue,Gross_Profit,Transactions,Has_Valid_Transactions,Cumulative_Net_Revenue,Rolling_3_Transaction_Days,Rolling_3_Calendar_Days
0,2026-01-05,"1,300.00",300.00,1,True,"1,300.00",NaN,"1,300.00"
1,2026-01-06,"1,425.00",325.00,1,True,"2,725.00",NaN,"1,362.50"
2,2026-01-07,250.00,130.00,1,True,"2,975.00",991.67,991.67
3,2026-01-08,486.00,96.00,1,True,"3,461.00",720.33,720.33
4,2026-01-09,440.00,120.00,1,True,"3,901.00",392.00,392.00
5,2026-01-10,665.00,145.00,1,True,"4,566.00",530.33,530.33
6,2026-01-11,225.00,120.00,1,True,"4,791.00",443.33,443.33
7,2026-01-12,"1,152.00",192.00,1,True,"5,943.00",680.67,680.67
8,2026-01-13,NaN,NaN,0,False,"5,943.00",NaN,688.50
9,2026-01-14,240.00,128.00,1,True,"6,183.00",539.00,696.00


In [44]:
# 3-4. Highest-revenue date and lowest POSITIVE-revenue date
observed = daily.dropna(subset=["Net_Revenue"])
print("Highest:", observed.loc[observed["Net_Revenue"].idxmax(), ["Date", "Net_Revenue"]].tolist())
positive = observed[observed["Net_Revenue"] > 0]
print("Lowest positive:", positive.loc[positive["Net_Revenue"].idxmin(), ["Date", "Net_Revenue"]].tolist())

Highest: [Timestamp('2026-01-15 00:00:00'), np.float64(1938.0)]
Lowest positive: [Timestamp('2026-01-17 00:00:00'), np.float64(216.0)]


In [45]:
# 7. Dates without recorded (valid) transactions - within observed range and across all of January
print("No valid transactions (5-24 Jan):", daily.loc[~daily["Has_Valid_Transactions"], "Date"].dt.strftime("%d %b").tolist())
all_jan = pd.date_range("2026-01-01", "2026-01-31", freq="D")
raw_dates = set(cleaning.parse_dates(sales_df["Date"]))
print("No records at all in January:", [d.strftime("%d %b") for d in all_jan if d not in raw_dates])

No valid transactions (5-24 Jan): ['13 Jan', '16 Jan', '19 Jan']
No records at all in January: ['01 Jan', '02 Jan', '03 Jan', '04 Jan', '25 Jan', '26 Jan', '27 Jan', '28 Jan', '29 Jan', '30 Jan', '31 Jan']


In [46]:
# 9. Compare equal-length reporting periods (10 calendar days each)
analytics.compare_periods(financial_df, {"5-14 Jan": ("2026-01-05", "2026-01-14"),
                                         "15-24 Jan": ("2026-01-15", "2026-01-24")})

,Period,Start,End,Calendar_Days,Days_With_Valid_Transactions,Transactions,Net_Revenue,Gross_Profit,Revenue_Per_Calendar_Day,Revenue_Per_Transaction_Day
0,5-14 Jan,2026-01-05,2026-01-14,10,9,9,"6,183.00","1,556.00",618.30,687.00
1,15-24 Jan,2026-01-15,2026-01-24,10,8,8,"7,157.50","1,699.50",715.75,894.69


**Interpretation — Stage 6.** Valid transactions run from 5 to 24 January, one per day.
* The highest-revenue date is **15 Jan ($1,938.00, TX011 Laptop)**. The lowest positive-revenue date is **17 Jan ($216.00)**.
* 13, 16 and 19 Jan have records, but all three are rejected, so they have **no valid transactions**. 1–4 Jan and 25–31 Jan have no records at all.
* These dates are kept as NaN (unknown), **not zero**. The cumulative total carries forward over them, and two rolling averages are shown: over the last 3 *transaction days* and over the last 3 *calendar days*.
* Comparing two equal 10-day periods: 15–24 Jan earned $7,157.50 against $6,183.00 for 5–14 Jan (+15.8%), with fewer valid transactions (8 vs 9).

With about one transaction a day, this is not evidence of a trend. The full-month total is also unknown, because 11 days of January are not covered.

## Stage 7 — Product and sales segmentation

In [47]:
# 1-2. High / Medium / Low value classification using tertile thresholds
segmented, thresholds = analytics.segment_transactions(financial_df)
print({k: round(v, 2) for k, v in thresholds.items()})
display(segmented["Value_Segment"].value_counts())
segmented[["Transaction_ID", "Product", "Net_Revenue", "Value_Segment"]].sort_values("Net_Revenue", ascending=False)

{'Low_below': 345.67, 'High_from': 1021.33}


Value_Segment
High      6
Low       6
Medium    5
Name: count, dtype: int64

,Transaction_ID,Product,Net_Revenue,Value_Segment
9,TX011,Laptop,"1,938.00",High
15,TX019,Smartphone,"1,782.00",High
11,TX014,Smartphone,"1,472.50",High
1,TX002,Smartphone,"1,425.00",High
0,TX001,Laptop,"1,300.00",High
7,TX008,Smartphone,"1,152.00",High
12,TX016,Monitor,760.00,Medium
5,TX006,Laptop,665.00,Medium
3,TX004,Monitor,486.00,Medium
4,TX005,Printer,440.00,Medium


In [48]:
# 3-6. High-performing / underperforming / low-margin products via a revenue x margin classification
product_class = analytics.classify_performance(product_perf, overall_margin=overall_margin)
product_class[["Product", "Net_Revenue", "Revenue_Share_Pct", "Gross_Profit", "Gross_Profit_Margin_Pct", "Performance_Class"]]

,Product,Net_Revenue,Revenue_Share_Pct,Gross_Profit,Gross_Profit_Margin_Pct,Performance_Class
0,Smartphone,"5,831.50",43.71,"1,096.50",18.80,Volume Driver
1,Laptop,"3,903.00",29.26,853.00,21.85,Volume Driver
2,Monitor,"1,246.00",9.34,296.00,23.76,Underperformer
3,Printer,877.00,6.57,217.00,24.74,Margin Niche
4,Keyboard,742.00,5.56,393.00,52.96,Margin Niche
5,Mouse,741.00,5.55,400.00,53.98,Margin Niche


In [49]:
# 7. Category classification
analytics.classify_performance(category_perf, overall_margin=overall_margin)[
    ["Category", "Net_Revenue", "Gross_Profit", "Gross_Profit_Margin_Pct", "Performance_Class"]]

,Category,Net_Revenue,Gross_Profit,Gross_Profit_Margin_Pct,Performance_Class
0,Mobile,"5,831.50","1,096.50",18.80,Volume Driver
1,Computers,"5,149.00","1,149.00",22.32,Volume Driver
2,Accessories,"1,483.00",793.00,53.47,Margin Niche
3,Office Equipment,877.00,217.00,24.74,Margin Niche


In [50]:
# 8. Unusually high discounts (>= 10%, the top tier observed)
high_disc = analytics.flag_high_discounts(financial_df)
high_disc.loc[high_disc["High_Discount"], ["Transaction_ID", "Product", "Salesperson", "Discount_Percent", "Discount_Amount", "Gross_Profit_Margin"]]

,Transaction_ID,Product,Salesperson,Discount_Percent,Discount_Amount,Gross_Profit_Margin
3,TX004,Monitor,John,10.00,54.00,19.75
7,TX008,Smartphone,Sarah,10.00,128.00,16.67
15,TX019,Smartphone,Sarah,10.00,198.00,15.82


In [51]:
# 9. Salesperson revenue and profit contributions
sp_perf[["Salesperson", "Revenue_Share_Pct", "Profit_Share_Pct"]]

,Salesperson,Revenue_Share_Pct,Profit_Share_Pct
0,Mary,31.78,30.49
1,Sarah,30.99,24.39
2,John,27.91,24.70
3,Peter,9.32,20.43


**Interpretation — Stage 7.** Thresholds are the tertiles of net revenue: **Low < $345.67 ≤ Medium < $1,021.33 ≤ High**. The bands don't overlap and every transaction is classified (6 / 5 / 6).
High-value transactions make up 68% of revenue but are all Laptop or Smartphone sales, at margins of about 16–23%.
On the revenue × margin matrix, Smartphone and Laptop are *Volume Drivers*, Keyboard, Mouse and Printer are *Margin Niches*, and Monitor is the only *Underperformer*.
The three 10% discounts (TX004, TX008, TX019) produced the three lowest margins in the dataset (15.8–19.8%). Two of them are Sarah's Smartphone sales.
Limitation: revenue-based segments ignore margin and volume, and with 17 records one transaction can move a threshold.

## Stage 8 — Multi-DataFrame operations

In [52]:
reference_raw = pd.DataFrame(get_product_reference())
reference_raw

,Product_Code,Product_Name,Product_Category,Reorder_Threshold
0,ZT-CMP-001,Laptop,Computers,5
1,ZT-CMP-002,Monitor,Computers,4
2,ZT-MOB-001,Smartphone,Mobile,6
3,ZT-ACC-001,keyboard,Accessories,10
4,ZT-ACC-002,Mouse,Accessories,15
5,ZT-OFF-001,Printer,Office Equipment,3
6,ZT-MOB-002,Tablet,Mobile,4


In [53]:
# 1-2. Validate product codes and standardise product names
reference = analytics.prepare_product_reference(reference_raw)
reference

,Product_Code,Product_Name,Product_Category,Reorder_Threshold
0,ZT-CMP-001,Laptop,Computers,5
1,ZT-CMP-002,Monitor,Computers,4
2,ZT-MOB-001,Smartphone,Mobile,6
3,ZT-ACC-001,Keyboard,Accessories,10
4,ZT-ACC-002,Mouse,Accessories,15
5,ZT-OFF-001,Printer,Office Equipment,3
6,ZT-MOB-002,Tablet,Mobile,4


In [54]:
# 3-4, 6. Left and inner joins with validate='many_to_one'
left = analytics.merge_sales_with_reference(financial_df, reference, how="left")
inner = analytics.merge_sales_with_reference(financial_df, reference, how="inner")
print("Rows  sales/left/inner:", len(financial_df), len(left), len(inner))
print("Revenue sales/left/inner:", financial_df["Net_Revenue"].sum(), left["Net_Revenue"].sum(), inner["Net_Revenue"].sum())
left["_merge"].value_counts()

Rows  sales/left/inner: 17 17 17
Revenue sales/left/inner: 13340.5 13340.5 13340.5


_merge
both          17
left_only      0
right_only     0
Name: count, dtype: int64

In [55]:
# 5. Unmatched records on either side (outer join)
analytics.unmatched_records(financial_df, reference)

,Transaction_ID,Product,Product_Code,Product_Name,_merge
17,<NA>,<NA>,ZT-MOB-002,Tablet,right_only


In [56]:
# Consistency: does the sales category agree with the reference category?
print("Category mismatches:", (left["Category"] != left["Product_Category"]).sum())
# 8. No duplicate transactions introduced
print("Duplicate Transaction_IDs after merge:", left["Transaction_ID"].duplicated().sum())

Category mismatches: 0
Duplicate Transaction_IDs after merge: 0


In [57]:
# What happens with a bad (duplicated) reference key: merge validation refuses it
bad_ref = pd.concat([reference, reference.iloc[[0]]], ignore_index=True)
try:
    financial_df.merge(bad_ref, left_on="Product", right_on="Product_Name", validate="many_to_one")
except pd.errors.MergeError as exc:
    print("MergeError raised as expected ->", exc)

MergeError raised as expected -> Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 Product_Name
      Laptop ...


In [58]:
# 7. City x category revenue pivot table
analytics.revenue_pivot(left, index="City", columns="Category")

Category,Accessories,Computers,Mobile,Office Equipment,Total
City,,,,,
Bulawayo,0.00,"3,724.00",0.00,0.00,"3,724.00"
Gweru,"1,243.00",0.00,0.00,0.00,"1,243.00"
Harare,240.00,665.00,"2,897.50",437.00,"4,239.50"
Mutare,0.00,760.00,"2,934.00",440.00,"4,134.00"
Total,"1,483.00","5,149.00","5,831.50",877.00,"13,340.50"


**Interpretation — Stage 8.** The reference table's product codes are valid and unique. Its names were standardised (`keyboard` and ` Mouse ` fixed) before joining on the standardised product name.
`validate="many_to_one"` enforces a unique key: a duplicated reference row is refused with a `MergeError`. The left and inner joins both return 17 rows and $13,340.50, so no transactions were duplicated or lost, and every sale matched.
The outer join shows one unmatched reference item, **Tablet (ZT-MOB-002)**. It is in the catalogue but has no sales in the sample.
Sales categories agree with the reference categories in all rows.

## Stage 9 — Advanced data manipulation and business intelligence

In [59]:
# 1. Multi-level aggregation by city and category
city_cat = financial_df.groupby(["City", "Category"]).agg(
    Transactions=("Transaction_ID", "count"), Net_Revenue=("Net_Revenue", "sum"), Gross_Profit=("Gross_Profit", "sum"))
city_cat["Margin_Pct"] = analytics.safe_margin(city_cat["Gross_Profit"], city_cat["Net_Revenue"])
city_cat

Transactions  Net_Revenue  Gross_Profit  Margin_Pct
City     Category                                                             
Bulawayo Computers                    3     3,724.00        804.00       21.59
Gweru    Accessories                  5     1,243.00        665.00       53.50
Harare   Accessories                  1       240.00        128.00       53.33
         Computers                    1       665.00        145.00       21.80
         Mobile                       2     2,897.50        622.50       21.48
         Office Equipment             1       437.00         97.00       22.20
Mutare   Computers                    1       760.00        200.00       26.32
         Mobile                       2     2,934.00        474.00       16.16
         Office Equipment             1       440.00        120.00       27.27

In [60]:
# 2. Above-average revenue transactions with below-average (overall) margin
avg_rev = financial_df["Net_Revenue"].mean()
financial_df[(financial_df["Net_Revenue"] > avg_rev) & (financial_df["Gross_Profit_Margin"] < overall_margin)][
    ["Transaction_ID", "Product", "City", "Net_Revenue", "Gross_Profit_Margin", "Discount_Percent"]]

,Transaction_ID,Product,City,Net_Revenue,Gross_Profit_Margin,Discount_Percent
0,TX001,Laptop,Bulawayo,"1,300.00",23.08,0.00
1,TX002,Smartphone,Harare,"1,425.00",22.81,5.00
7,TX008,Smartphone,Mutare,"1,152.00",16.67,10.00
9,TX011,Laptop,Bulawayo,"1,938.00",21.05,5.00
11,TX014,Smartphone,Harare,"1,472.50",20.20,5.00
15,TX019,Smartphone,Mutare,"1,782.00",15.82,10.00


In [61]:
# 3. Rank salespeople within each city on AGGREGATED revenue (not individual transactions)
sp_city = financial_df.groupby(["City", "Salesperson"], as_index=False)["Net_Revenue"].sum()
sp_city["Rank_In_City"] = sp_city.groupby("City")["Net_Revenue"].rank(ascending=False, method="min").astype(int)
sp_city

,City,Salesperson,Net_Revenue,Rank_In_City
0,Bulawayo,John,"3,724.00",1
1,Gweru,Peter,"1,243.00",1
2,Harare,Mary,"4,239.50",1
3,Mutare,Sarah,"4,134.00",1


In [62]:
# 4. Category-by-city pivot table (pivot_table aggregates; pivot() would fail on duplicate pairs)
pd.pivot_table(financial_df, index="Category", columns="City", values="Net_Revenue", aggfunc="sum", fill_value=0, margins=True, margins_name="Total")

City,Bulawayo,Gweru,Harare,Mutare,Total
Category,,,,,
Accessories,0.00,"1,243.00",240.00,0.00,"1,483.00"
Computers,"3,724.00",0.00,665.00,760.00,"5,149.00"
Mobile,0.00,0.00,"2,897.50","2,934.00","5,831.50"
Office Equipment,0.00,0.00,437.00,440.00,877.00
Total,"3,724.00","1,243.00","4,239.50","4,134.00","13,340.50"


In [63]:
# 5. Performance classifications for cities and salespeople
display(analytics.classify_performance(city_perf.dropna(subset=["Net_Revenue"]), overall_margin=overall_margin)[["City", "Net_Revenue", "Gross_Profit_Margin_Pct", "Performance_Class"]])
analytics.classify_performance(sp_perf, overall_margin=overall_margin)[["Salesperson", "Net_Revenue", "Gross_Profit_Margin_Pct", "Performance_Class"]]

,City,Net_Revenue,Gross_Profit_Margin_Pct,Performance_Class
0,Harare,"4,239.50",23.41,Volume Driver
1,Mutare,"4,134.00",19.21,Volume Driver
2,Bulawayo,"3,724.00",21.59,Volume Driver
3,Gweru,"1,243.00",53.50,Margin Niche


,Salesperson,Net_Revenue,Gross_Profit_Margin_Pct,Performance_Class
0,Mary,"4,239.50",23.41,Volume Driver
1,Sarah,"4,134.00",19.21,Volume Driver
2,John,"3,724.00",21.59,Volume Driver
3,Peter,"1,243.00",53.50,Margin Niche


In [64]:
# 6. Cumulative revenue separately for each city (sorted chronologically first)
by_city = financial_df.sort_values(["City", "Date"]).copy()
by_city["City_Cumulative_Revenue"] = by_city.groupby("City")["Net_Revenue"].cumsum()
by_city[["City", "Date", "Transaction_ID", "Net_Revenue", "City_Cumulative_Revenue"]]

,City,Date,Transaction_ID,Net_Revenue,City_Cumulative_Revenue
0,Bulawayo,2026-01-05,TX001,"1,300.00","1,300.00"
3,Bulawayo,2026-01-08,TX004,486.00,"1,786.00"
9,Bulawayo,2026-01-15,TX011,"1,938.00","3,724.00"
2,Gweru,2026-01-07,TX003,250.00,250.00
6,Gweru,2026-01-11,TX007,225.00,475.00
10,Gweru,2026-01-17,TX013,216.00,691.00
13,Gweru,2026-01-21,TX017,252.00,943.00
16,Gweru,2026-01-24,TX020,300.00,"1,243.00"
1,Harare,2026-01-06,TX002,"1,425.00","1,425.00"
5,Harare,2026-01-10,TX006,665.00,"2,090.00"


In [65]:
# 7. Product revenue contribution within each category (transform keeps index alignment)
prod_cat = financial_df.groupby(["Category", "Product"], as_index=False)["Net_Revenue"].sum()
prod_cat["Share_Within_Category_Pct"] = prod_cat["Net_Revenue"] / prod_cat.groupby("Category")["Net_Revenue"].transform("sum") * 100
prod_cat

,Category,Product,Net_Revenue,Share_Within_Category_Pct
0,Accessories,Keyboard,742.00,50.03
1,Accessories,Mouse,741.00,49.97
2,Computers,Laptop,"3,903.00",75.80
3,Computers,Monitor,"1,246.00",24.20
4,Mobile,Smartphone,"5,831.50",100.00
5,Office Equipment,Printer,877.00,100.00


In [66]:
# 8. Three most profitable categories
category_perf.nlargest(3, "Gross_Profit")[["Category", "Gross_Profit", "Gross_Profit_Margin_Pct"]]

,Category,Gross_Profit,Gross_Profit_Margin_Pct
1,Computers,"1,149.00",22.32
0,Mobile,"1,096.50",18.80
2,Accessories,793.00,53.47


In [67]:
# 9. Reconcile grouped and transaction-level revenue
checks = pd.DataFrame({
    "Source": ["Transactions", "City x Category", "Salesperson x City", "Product x Category"],
    "Net_Revenue": [financial_df["Net_Revenue"].sum(), city_cat["Net_Revenue"].sum(),
                    sp_city["Net_Revenue"].sum(), prod_cat["Net_Revenue"].sum()]})
checks["Reconciled"] = np.isclose(checks["Net_Revenue"], financial_df["Net_Revenue"].sum())
checks

,Source,Net_Revenue,Reconciled
0,Transactions,"13,340.50",True
1,City x Category,"13,340.50",True
2,Salesperson x City,"13,340.50",True
3,Product x Category,"13,340.50",True


In [68]:
# 10. Compare original and processed datasets
pd.DataFrame({
    "Original": [len(sales_df), sales_df.shape[1], sales_df["Product"].nunique(), str(sales_df["Date"].dtype),
                 int(sales_df.isna().sum().sum())],
    "Processed (accepted)": [len(financial_df), financial_df.shape[1], financial_df["Product"].nunique(),
                             str(financial_df["Date"].dtype), int(financial_df[validation.MANDATORY_COLUMNS].isna().sum().sum())],
}, index=["Rows", "Columns", "Distinct product names", "Date dtype", "Missing mandatory values"])

,Original,Processed (accepted)
Rows,20,17
Columns,11,24
Distinct product names,9,6
Date dtype,str,datetime64[us]
Missing mandatory values,1,0


**Interpretation — Stage 9.**
* **City × category:** Bulawayo sells only Computers and Gweru only Accessories. Harare is the most diversified city (4 categories).
* **Big sales at weak margins:** six transactions earn above-average revenue at below-average margin. All are Laptop or Smartphone sales, and all but one were discounted. This is the main pressure on overall margin.
* **Rankings within cities** are trivial (one salesperson per city), which again shows the confounding.
* **Cumulative revenue** is calculated per city after sorting by date. Product shares within each category use `transform`, so the index stays aligned.
* **Top 3 categories by gross profit:** Computers, Mobile and Accessories.

All grouped totals reconcile to the transaction total. Compared with the original, the processed data has 17 rows (+3 quarantined), 6 distinct product names instead of 8, datetime dates and no missing mandatory values.

## Stages 10–11 — Automated tests and the reusable pipeline
Tests live in `tests/test_sales_pipeline.py` (run `python -m pytest -v` from the repository root).
Below, the complete pipeline is executed from scratch and its results are checked against the notebook figures.

In [69]:
results = run_pipeline()   # builds, validates, assesses, cleans, transforms, aggregates, exports
print("Files written:", [p.name for p in results["files"]])
print("Pipeline net revenue matches notebook:", np.isclose(results["financial"]["Net_Revenue"].sum(), financial_df["Net_Revenue"].sum()))

Files written: ['raw_sales_original.csv', 'data_quality_report.csv', 'data_quality_rule_summary.csv', 'cleaning_summary.csv', 'cleaned_sales.csv', 'rejected_records.csv', 'sales_summary.csv', 'city_performance.csv', 'product_performance.csv', 'category_performance.csv', 'salesperson_performance.csv', 'payment_method_performance.csv', 'transaction_segments.csv', 'daily_revenue.csv']
Pipeline net revenue matches notebook: True


In [70]:
# Demonstration: a deliberately invalid record is caught by the validation process
bad = get_sales_data()
for key, value in {"Transaction_ID": "TX999", "Date": "31/02/2026", "Product": "Laptop", "Category": "Computers",
                   "Quantity": -5, "Unit_Price": None, "Unit_Cost": 500, "Salesperson": "John", "City": "Harare",
                   "Payment_Method": "Cash", "Discount_Percent": 150}.items():
    bad[key].append(value)
demo = run_pipeline(source_dict=bad, export=False)
demo["cleaning"].rejected.loc[demo["cleaning"].rejected["Transaction_ID"] == "TX999", ["Transaction_ID", "Review_Reason"]]

,Transaction_ID,Review_Reason
3,TX999,Invalid or unparseable date; Negative quantity...


In [71]:
import subprocess
completed = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"], cwd=PROJECT_ROOT, capture_output=True, text=True)
print(completed.stdout[-600:])

............................                                             [100%]
28 passed in 1.15s



**Final answer to the management challenge** is in `reports/management_report.md`.